In [7]:
import os
from langchain.chat_models import  init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("hey")
response.content


'<think>\nOkay, the user sent "hey". That\'s pretty open-ended. I need to figure out what they need help with. Let me start by greeting them back and asking how I can assist. Maybe they have a specific question or need information on a topic. I should keep my response friendly and encouraging so they feel comfortable to elaborate. Let me make sure to offer help with anything from general knowledge to problem-solving. Yeah, that should cover it.\n</think>\n\nHello! How can I assist you today? 😊'

In [9]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """get the current weather of the location"""
    return f"the current weather of the {location} is sunny!"

model_with_tool = model.bind_tools([get_weather])
answer = model_with_tool.invoke("what is the weather of surat?")
answer
for tool_call in answer.tool_calls:
    print(f"tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")

tool: get_weather
args: {'location': 'Surat'}


### tool execution loop


In [20]:
# first get tool call

messages = [{"role": "user", "content": "what is the weather in surat?"}]
ai_message = model_with_tool.invoke(messages)
messages.append(ai_message)

# second introduce our defined tool to tool call
for tool_call in ai_message.tool_calls:
    result = get_weather.invoke(tool_call)
    messages.append(result)

# combine tool call and our custome tool and generate the output
final_response = model_with_tool.invoke(messages)
print(final_response.text)


The current weather in Surat, India is **sunny**! 🌞 Let me know if you need more details.
